# Pipeline: search, classify, and reconstruct a factory

A working pipeline through this repo's pieces: run the two-group and/or symmetry-free search for
a chosen parameter set, classify what comes out (`degree`, `essential_dim`, `t_count`,
`decomposition`), and reconstruct the explicit circuit as a binary matrix or a real Quirk link.
Run from the repo root.

For what each parameter means, see [`searches/README.md`](searches/README.md) (search parameters)
and [`classification/README.md`](classification/README.md) (classification/circuit output). For
the paper and overall findings, see the root [`README.md`](README.md).


In [ ]:
import sys, os, csv
from itertools import combinations

ROOT = os.path.abspath(".")
sys.path.insert(0, os.path.join(ROOT, "searches"))
sys.path.insert(0, os.path.join(ROOT, "classification"))

import Two_group as tg
import symfree_search as sym
import classify
import quirk

print("modules loaded: Two_group, symfree_search, classify, quirk")


modules loaded: Two_group, symfree_search, classify, quirk


## 1. Two-group search

Parameters: level `l`, circuit size `n`, output count `k`, skip parameters `s_total`, `s_O`
(`s_S` is fixed at 1). See `searches/README.md` for what these mean; every row of
`outputs/factory_catalogue_l{2,3,4}.csv` lists the exact `(l,n,k,s_total,s_O)` that produced it.


In [ ]:
l, n, k, s_total, s_O = 3, 4, 2, 1, 1   # -> [[12,2,2]], a CS factory

n_minus_k = n - k
pairs, W_total, W_O = tg.build_gate_set(k, n_minus_k, s_total, s_O)
valid_signs = tg.find_valid_signs(l, n, k, pairs)
print(f"{len(valid_signs)} valid sign assignment(s)" if valid_signs else "no valid borrowed identity at these parameters")

sigma_dict = valid_signs[0]
N = tg.extract_factory(k, n_minus_k, sigma_dict, W_O, W_total)

# reconstruct the actual gate list (needed to classify / print the circuit below)
removed = {(w, 0) for w in (set(W_O) & set(W_total)) if w >= 1}
out_wires, chk_wires = list(range(k)), list(range(k, n))
Gf, cf = [], []
for (wO, wS), sig in sigma_dict.items():
    if sig == 0 or (wO, wS) in removed:
        continue
    for Osub in combinations(out_wires, wO):
        for Ssub in combinations(chk_wires, wS):
            Gf.append(frozenset(Osub + Ssub))
            cf.append(sig)
assert len(Gf) == N

factory_tg = dict(Gf=Gf, cf=cf, out=out_wires, n=n, l=l,
                  label=f"two-group  l={l} n={n} k={k} s_total={s_total} s_O={s_O}")
print(f"[[{N},{k},2]]  ({factory_tg['label']})")


1 valid sign assignment(s)
[[12,2,2]]  (two-group  l=3 n=4 k=2 s_total=1 s_O=1)


## 2. Symmetry-free search

Parameters: level `l`, output block-size partition `parts` (e.g. `(3, 2)` = one weight-≤3 block
and one weight-≤2 block), and number of check qubits. Unlike two-group, this search has no
built-in symmetry across outputs, so it can produce **mixed-output** factories (different gate
types in different blocks of the same factory).


In [ ]:
l_sf, parts, checks = 3, (3, 2), 2   # -> [[18,5,2]], mixed CS+CCZ output

result = sym.solve_binding(list(parts), checks, l_sf)
if result is None:
    print(f"no borrowed identity at l={l_sf} parts={parts} checks={checks}")
else:
    Gf_sf = [supp for supp, _ in result["gates"]]
    cf_sf = [c for _, c in result["gates"]]
    out_sf = list(result["O"])
    factory_sf = dict(Gf=Gf_sf, cf=cf_sf, out=out_sf, n=result["n"], l=l_sf,
                      label=f"symmetry-free  l={l_sf} parts={parts} checks={checks}")
    print(f"[[{len(Gf_sf)},{len(out_sf)},2]]  ({factory_sf['label']})")


[[18,5,2]]  (symmetry-free  l=3 parts=(3, 2) checks=2)


## 3. Classify a factory's deposited output

Pick which factory to classify by setting `factory` to `factory_tg` or `factory_sf` (from the two
cells above), or to a factory loaded from the catalogue (section 6). `degree`/`essential_dim` are
exact for `k≤4`, a documented upper bound for `k≥5`; `t_count` is exact but only defined at `l=3`.
See `classification/README.md` for what `decomposition`'s qubit-indexed labels mean.


In [ ]:
factory = factory_tg   # <-- change to factory_sf to classify the other one

info = classify.classify(factory["Gf"], factory["cf"], factory["out"], factory["l"])
print(factory["label"])
print(f"  degree        = {info['degree']}" + (f"  [{info['degree_note']}]" if info["degree_note"] else "  (exact)"))
print(f"  essential_dim = {info['essential_dim']}  (of k={len(factory['out'])} output qubits)")
if factory["l"] == 3:
    print(f"  t_count       = {info['t_count']}" + (f"  [{info['t_count_note']}]" if info["t_count_note"] else "  (exact)"))
else:
    print("  t_count       = only defined at l=3")
print(f"  decomposition = {info['decomposition']}")


two-group  l=3 n=4 k=2 s_total=1 s_O=1
  degree        = 2  (exact)
  essential_dim = 2  (of k=2 output qubits)
  t_count       = 3  (exact)
  decomposition = CS01


## 4. Explicit circuit as a binary matrix

Rows = wires (outputs first, then checks), columns = gates — the same column convention as the
`sj-magic-state-factory-searches` companion repo's `master_catalog` catalogue.
`classification/export_circuit.py` does this as a standalone CLI if you'd rather not use the
notebook.


In [ ]:
def print_matrix(factory):
    Gf, n = factory["Gf"], factory["n"]
    print(f"n={n} wires ({len(factory['out'])} outputs + {n - len(factory['out'])} checks), N={len(Gf)} gates")
    for wire in range(n):
        print(" " + "".join("1" if wire in g else "0" for g in Gf))

print_matrix(factory)


n=4 wires (2 outputs + 2 checks), N=12 gates
 000110010111
 000001101111
 101101011101
 011010111011


## 5. A real, clickable Quirk circuit

`classification/quirk.py` renders the same gate list as an actual [Quirk](https://algassert.com/quirk)
circuit: each multi-qubit parity rotation becomes a CNOT-ladder onto one "hub" qubit, a
single-qubit phase gate there, then the ladder undone (no ancillas). Verified by direct unitary
simulation against the target diagonal (see `classification/quirk.py`'s module docstring) — not
just asserted. Pick a small factory before running this (the URL grows with column count, which
is roughly `N` times the largest gate weight).


In [ ]:
url = quirk.quirk_url(factory["Gf"], factory["cf"], factory["n"], factory["l"])
print(f"{len(quirk.circuit_to_quirk_cols(factory['Gf'], factory['cf'], factory['n'], factory['l']))} Quirk columns")
print(url)


44 Quirk columns
https://algassert.com/quirk#circuit=%7B%22cols%22%3A%20%5B%5B%221%22%2C%20%221%22%2C%20%22Z%5E%5Cu00bc%22%2C%20%221%22%5D%2C%20%5B%221%22%2C%20%221%22%2C%20%221%22%2C%20%22Z%5E%5Cu00bc%22%5D%2C%20%5B%221%22%2C%20%221%22%2C%20%22X%22%2C%20%22%5Cu2022%22%5D%2C%20%5B%221%22%2C%20%221%22%2C%20%22Z%5E%5Cu00bc%22%2C%20%221%22%5D%2C%20%5B%221%22%2C%20%221%22%2C%20%22X%22%2C%20%22%5Cu2022%22%5D%2C%20%5B%22X%22%2C%20%221%22%2C%20%22%5Cu2022%22%2C%20%221%22%5D%2C%20%5B%22Z%5E%5Cu00bc%22%2C%20%221%22%2C%20%221%22%2C%20%221%22%5D%2C%20%5B%22X%22%2C%20%221%22%2C%20%22%5Cu2022%22%2C%20%221%22%5D%2C%20%5B%22X%22%2C%20%221%22%2C%20%221%22%2C%20%22%5Cu2022%22%5D%2C%20%5B%22Z%5E%5Cu00bc%22%2C%20%221%22%2C%20%221%22%2C%20%221%22%5D%2C%20%5B%22X%22%2C%20%221%22%2C%20%221%22%2C%20%22%5Cu2022%22%5D%2C%20%5B%221%22%2C%20%22X%22%2C%20%22%5Cu2022%22%2C%20%221%22%5D%2C%20%5B%221%22%2C%20%22Z%5E%5Cu00bc%22%2C%20%221%22%2C%20%221%22%5D%2C%20%5B%221%22%2C%20%22X%22%2C%20%22%5Cu2022%22%2C%20%221%22

## 6. Independent re-verification

A circuit is a valid borrowed identity iff no genuine (level-`l`) content touches a check qubit:
every monomial that includes a check wire must have an *even* coefficient mod `2^size`, for every
size up to `l` (Theorem 5's own `i ∈ {1,...,l}` range — beyond size `l` the prefactor already
forces this). This re-derives validity straight from the gate list, independently of whichever
search produced it.


In [ ]:
def verify_borrowed_identity(factory):
    Gf, cf, out, n, l = factory["Gf"], factory["cf"], factory["out"], factory["n"], factory["l"]
    checks = [q for q in range(n) if q not in out]
    for size in range(1, l + 1):
        for T in combinations(range(n), size):
            if not (set(T) & set(checks)):
                continue
            c = classify.coeff(Gf, cf, T, l)
            if c % (1 << size):
                return False, T, c
    return True, None, None

ok, bad_T, bad_c = verify_borrowed_identity(factory)
print("valid borrowed identity:", ok, "" if ok else f"(fails at T={bad_T}, coeff={bad_c})")


valid borrowed identity: True 


## 7. Load an existing factory from the catalogue instead

Every row in `outputs/factory_catalogue_l{2,3,4}.csv` already carries the parameters to
regenerate it (see `outputs/README.md`) — useful for exploring what's already found before running
a fresh search.


In [ ]:
target_l, target_N, target_k = 3, 18, 5

with open(f"outputs/factory_catalogue_l{target_l}.csv") as f:
    rows = [r for r in csv.DictReader(f) if int(r["N"]) == target_N and int(r["k"]) == target_k]
print(f"{len(rows)} matching row(s) for [[{target_N},{target_k},2]] at l={target_l}")

row = rows[0]
print({k: v for k, v in row.items() if v != ""})

if row["s_total"]:
    print(f"-> classification/export_circuit.py two-group --l {target_l} --n {row['n']} "
          f"--k {target_k} --s_total {row['s_total']} --s_O {row['s_O']}")
else:
    print(f"-> classification/export_circuit.py symfree --l {target_l} "
          f"--parts {row['parts'].replace(chr(43), chr(44))} --checks {int(row['n']) - target_k}")


1 matching row(s) for [[18,5,2]] at l=3
{'l': '3', 'N': '18', 'k': '5', 'd': '2', 'degree': '3', 'essential_dim': '5', 't_count': '9', 'decomposition': 'CCZ012+CS34', 'n': '7', 'parts': '3+2', 'search': 'symmetry-free'}
-> classification/export_circuit.py symfree --l 3 --parts 3,2 --checks 2
